# STAGE 1 RESTART — clean, verified, provenance-safe  (heat = TFF)

Completes the drift-budget law's ‖ḡ‖ factor on solid, persisted, correctly-
labeled data. **Protocol lock:** the E1 stream is one corruption's severity-5
split **cycled ×15** (~2355 steps — the continual horizon; validated:
gaussian ×15, p=0, η=1e-3 → first NaN = 601). Single-pass 157-step streams are
FORBIDDEN for law measurement. HARD criterion primary; soft recorded
separately. Provenance guards: Drive-path + sentinel check before any run,
per-run protocol fingerprints (incl. checkpoint sha256), consumed-file
manifests in every analysis artifact, end-of-campaign file-count check,
VOID rule (no reference ⇒ no curve verdict).

Cost estimate: 12 cells × ~6–9 runs × (~2355 steps ≈ 8–12 min A100) ≈
**8–14 h A100 total** — fully resumable; calibration corruptions run first so
the slope lands early. Run in tranches if needed (just re-run the campaign
cell; finished runs are skipped).

## 1. GPU check

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else "WARNING: no GPU — do not run the campaign on CPU.")

## 2. Config — `# === EDIT ME ===`

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = "https://github.com/octadion/heat.git"
REPO_DIR   = "heat"
GIT_BRANCH = "formulation"        # branch that contains the Stage-1 v2 scripts

RESULTS_DIR = "/content/drive/MyDrive/pstar_results"   # MUST be Drive-backed
CKPT_WRN    = "/content/drive/MyDrive/heat/experiments/checkpoints/wrn28_10_final.pt"
C10C_ROOT   = "data/cifar10c"
SEED, SEV, BATCH, WORKERS = 42, 5, 64, 2
CYCLES      = 15                  # LOCKED protocol field (recorded everywhere)
BISECT      = 3
# =======================================================================
print("RESULTS_DIR =", RESULTS_DIR, "| CYCLES =", CYCLES)

## 3. Mount Drive, clone repo, data, checkpoint

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(REPO_DIR):
    cmd = ["git", "clone"] + (["--branch", GIT_BRANCH] if GIT_BRANCH else []) + [REPO_URL, REPO_DIR]
    subprocess.run(cmd, check=True)
os.chdir("/content/" + REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["PYTHONUTF8"] = "1"

subprocess.run(["python", "scripts/download_cifar10c.py", "--root", C10C_ROOT], check=True)
assert os.path.exists(CKPT_WRN), f"checkpoint not found: {CKPT_WRN}"
import hashlib
print("cwd =", REPO_ROOT)
print("ckpt sha256[:16] =", hashlib.sha256(open(CKPT_WRN,'rb').read()).hexdigest()[:16])

## STEP 0 — recover or declare (read-only, ~seconds)

Strict-regex sweep of `RESULTS_DIR` for genuine single-corruption / forward
run JSONs; prints `n_steps` + recorded args for any found, or declares
"restarting from scratch". (The campaign script repeats this automatically.)

In [ ]:
import sys
sys.path.insert(0, REPO_ROOT)
from scripts import stage1v2_common as v2
v2.recover_scan(RESULTS_DIR)

## STEP 1–3 — the campaign (guards + gate + cycled E1; resumable)

`run_stage1_e1v2.py` enforces, in order: Drive-path guard + sentinel →
checkpoint sha256 → STEP-0 scan → **continual sanity gate (must exit 0)** →
cells `{gaussian_noise, elastic_transform | calibration} +
{impulse_noise, contrast | held-out}` × η ∈ {5e-4, 1e-3, 2e-3}, each cell
source → p_ref ladder → η-scaled p-grid → upward extension → bisection, HARD
primary. Every run JSON gets an embedded protocol fingerprint; the campaign
ends with the Drive-side file-count check. Re-run this cell after any
disconnect — finished runs are skipped.

In [ ]:
import subprocess
rc = subprocess.run([
    "python", "scripts/run_stage1_e1v2.py",
    "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT_WRN,
    "--c10c-root", C10C_ROOT, "--cycles", str(CYCLES),
    "--bisect-steps", str(BISECT), "--seed", str(SEED),
    "--batch-size", str(BATCH), "--num-workers", str(WORKERS),
]).returncode
print("\n[campaign] exit code", rc)
if rc != 0:
    print("[campaign] ABORTED by a guard or the gate — read the message above; "
          "do NOT override guards casually.")

## STEP 4 — analyses E1–E5 + verdict (zero GPU)

Fits (hard primary, soft separate) + continual reference recomputed on-disk
(expected hard-only ~1.152 ± 0.022, labeled *continual*), E2 cycle-aware
convergence, E3 coupling, E4 zero-shot with **S re-frozen fresh from this
campaign's calibration cells** (nothing from voided runs), E5 R-sensitivity.
All artifacts embed consumed-file manifests; VOID if no reference.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/analyze_stage1_v2.py",
                "--results-dir", RESULTS_DIR,
                "--cycles", str(CYCLES), "--seed", str(SEED)], check=False)

### Verdict + plots inline

In [ ]:
import os
from IPython.display import Image, Markdown, display
adir = os.path.join(RESULTS_DIR, "analysis")
for name in ("stage1v2_verdict.md",):
    p = os.path.join(adir, name)
    if os.path.exists(p):
        display(Markdown(open(p, encoding="utf-8").read()))
for name in ("stage1v2_gbar_law.png", "stage1v2_convergence.png"):
    p = os.path.join(adir, name)
    if os.path.exists(p):
        display(Image(filename=p))

## STEP 5 — STOP

Report the verdict block above, the `[guard c]` Drive file-count line from the
campaign output, and any deviations. **Stage 1b (E6/E7) and Stage 2 await
instruction; they will use `analysis/stage1v2_S_frozen.json` from THIS
campaign as the only reference.**